In [126]:
import gurobipy as gp
from gurobipy import GRB
from utils.data import *
import math
import warnings
import plotly.express as px

### Defining environment parameters

In [ ]:
start = dt.datetime(year=2000, month=1, day=1)
end = dt.datetime(year=2021, month=1, day=1)

all_stocks = ["NVDA", "AAPL", 'GOOG', "AVGO", "WMT", "JPM", "LLY", "XOM", "KO"]

# CHOOSE OBJECTIVE:
# -`min_var`    : Minimize expected variance
# -`max_mu`     : Maximize expected returns
# -`max_sharpe` : Maximize expected ratio of returns to variance

mode = "min_var"

n = len(all_stocks)

# COVARIATES (Inflation, risk-free rate)
#a = (0.005, 0.25)  # 2% yearly inflation, 0.25% risk free rate (represents circa 2015 "the good times")
a = (0.022, 4.5)  # 9% yearly inflation, 4.5% risk free rate (represents post-covid inflation period)
#a = (0.005, 3)  # 2% yearly inflation, 3% risk free rate (represents ideal federal reserve conditions)

T = 1/4
r = (1+a[1]/100)**(1/4)-1
S = [float(yf.Ticker(s).history(period="1d").Open.iloc[0]) for s in all_stocks]

### Calculating conditional moments

In [220]:
pce = load_pce()
effr = load_ffr()


mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)

new_sigma, sep_Sigma = add_covariates_to_covar(Sigma, all_stocks, [pce, effr], start, end)
new_mu, sep_mu = add_covariates_to_mu(mu, [pce, effr])

mu, Sigma = conditional_moments(sep_mu, sep_Sigma, a)

### Defining and solving model

In [221]:
m = gp.Model()

m.setParam("MIPGap", 0.01)
m.setParam("TimeLimit", 60)

S = [yf.Ticker(s).history(period="1d").Close.to_list()[0] for s in all_stocks]
sigma = [np.sqrt(Sigma[i, i]) for i in range(n)]

w = m.addMVar(n, lb=0, name="weight")

mu_d = m.addMVar(n, lb=-GRB.INFINITY, name="expectation")
Delta = m.addMVar(n, lb=0.1, ub=.5, name="Delta")
P = m.addMVar(n, lb=-GRB.INFINITY, name="put price")

x_pts_CDF = np.linspace(-5, 5, 101)
y_pts_CDF = np.array([math.erf(x/np.sqrt(2))*1/2+1/2 for x in x_pts_CDF])

x_pts_PDF = np.linspace(-5, 5, 101)
y_pts_PDF = np.array([math.exp(-x**2/2)*1/np.sqrt(2*np.pi) for x in x_pts_CDF])

# COMPUTING PRICE OF PUT OPTION & TRUNCATED EXPECTATION
for i in range(n):
    Phi_inv_expr = 4/1.7*(Delta[i] - 0.5)
    
    K_expr = (sigma[i]*Phi_inv_expr + mu[i] + 1)*S[i]

    d1 = m.addVar(lb=-GRB.INFINITY, name=f"d1_{i}")
    d2 = m.addVar(lb=-GRB.INFINITY, name=f"d2_{i}")

    m.addConstr(d1 == (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T)))
    m.addConstr(d2 == d1 - sigma[i]*np.sqrt(T))

    cdf_d1_expr = m.addVar(lb=-GRB.INFINITY, name=f"cdf_d1_{i}")
    cdf_d2_expr = m.addVar(lb=-GRB.INFINITY, name=f"cdf_d2_{i}")

    cdf_d1 = m.addGenConstrPWL(d1, cdf_d1_expr, x_pts_CDF,\
                                                y_pts_CDF)
    cdf_d2 = m.addGenConstrPWL(d2, cdf_d2_expr, x_pts_CDF,\
                                                y_pts_CDF)

    d1_expr = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
    d2_expr =  d1_expr - sigma[i]*np.sqrt(T)
    cdf_d1_expr = (0.5 - (1/np.sqrt(2*np.pi))*d1_expr)
    cdf_d2_expr = 0.5 - (1/np.sqrt(2*np.pi))*d2_expr
    
    phi = 1/(np.sqrt(2*np.pi))*(Phi_inv_expr**2/2)

    P_expr = K_expr*np.exp(-r*T)*cdf_d2_expr - S[i]*cdf_d1_expr

    m.addConstr(P[i] == P_expr)
    m.addConstr(mu_d[i] == mu[i] + sigma[i]*(Delta[i]*Phi_inv_expr+ phi) - P[i]/S[i])

# PORTFOLIO MUST BE INVESTED
m.addConstr(gp.quicksum([wi for wi in w]) == 1)

Sigma_d = np.zeros((n, n), dtype=gp.Var)

# COMPUTING TRUNCATED COVARIANCE MATRIX
for i in range(n):
    a_i = 4/1.7*(Delta[i] - 0.5)
    cdf_eta_i = m.addVar(lb=-GRB.INFINITY, name=f"cdf_eta_{i}")

    eta_i = m.addVar(lb=-GRB.INFINITY, name=f"eta_{i}")

    m.addConstr(eta_i == mu[i]/sigma[i]-a_i)

    cdf_etai = m.addGenConstrPWL(eta_i, cdf_eta_i, x_pts_CDF,\
                                                y_pts_CDF)
    phi_eta_i = m.addVar(lb=-GRB.INFINITY, name=f"phi_{i}")

    pdf_eta_i = m.addGenConstrPWL(eta_i, phi_eta_i, x_pts_PDF,\
                                            y_pts_PDF)
    
    # COVARIANCE TERM (SIGMA[i, j])
    for j in range(i+1, n):
        rho_ij = Sigma[i,j] / (sigma[i]*sigma[j])
        sqrt_1mrho2 = np.sqrt(1 - rho_ij**2)

        a_j = 4/1.7*(Delta[j] - 0.5)

        cdf_eta_j = m.addVar(lb=-GRB.INFINITY, name=f"cdf_eta_{i}")

        eta_j = m.addVar(lb=-GRB.INFINITY, name=f"eta_{i}")

        m.addConstr(eta_j == mu[j]/sigma[j]-a_j)
        
        cdf_etaj = m.addGenConstrPWL(eta_j, cdf_eta_j, x_pts_CDF,\
                                                    y_pts_CDF)
        
        phi_eta_j = m.addVar(lb=-GRB.INFINITY, name=f"phi_{j}")
       
        pdf_eta_j = m.addGenConstrPWL(eta_j, phi_eta_j, x_pts_PDF,\
                                                y_pts_PDF)

        w_ij = (eta_i - rho_ij*eta_j) / sqrt_1mrho2
        w_ji = (eta_j - rho_ij*eta_i) / sqrt_1mrho2

        Phi_w_ij = 0.5 + 1/np.sqrt(2*np.pi)*w_ij
        Phi_w_ji = 0.5 + 1/np.sqrt(2*np.pi)*w_ji

        Phi2 = cdf_eta_i*cdf_eta_j + rho_ij*phi_eta_i*phi_eta_j

        phi2 = phi_eta_i*phi_eta_j / sqrt_1mrho2        

        covar_ij = m.addVar(lb=-GRB.INFINITY, name=f"covar_{i}_{j}")

        m.addConstr(
            covar_ij == mu[i]*mu[j]+(rho_ij
            + (rho_ij * a_i * phi_eta_i * Phi_w_ji
            + rho_ij * a_j * phi_eta_j * Phi_w_ij
            + (1 - rho_ij**2) * phi2))*sigma[i]*sigma[j]/Phi2 -  mu_d[i]*mu_d[j]
        )

        Sigma_d[i, j] = covar_ij      
        Sigma_d[j, i] = covar_ij
 
    # VARIANCE TERM (SIGMA[i, i])
    var_i = m.addVar(lb=-GRB.INFINITY, name=f"var_{i}")

    m.addConstr(
        var_i == sigma[i]**2 *                                  \
        (1-phi_eta_i**2/(cdf_eta_i)**2 - eta_i*phi_eta_i/(cdf_eta_i))
    )

    Sigma_d[i, i] = var_i

# CONSTRAINTS TO COMPUTE RETURN/RISK RATIO EFFICIENTLY
t = m.addVar(lb=-GRB.INFINITY)
var_norm = m.addVar()
var_norm_sq = m.addVar()

m.addConstr(var_norm_sq == gp.quicksum([Sigma_d[i, i] * w[i]**2 for i in range(n)]) + \
                        2*gp.quicksum([Sigma_d[i, j] * w[i]*w[j] for i in range(n) for j in range(n) if j > i]))
m.addConstr(var_norm_sq == var_norm**2)

m.addConstr(gp.quicksum([mu_d[i]*w[i] for i in range(n)]) -r >= var_norm * t)

if mode == "max_mu":   
    m.setObjective(gp.quicksum(mu_d[i]*w[i] for i in range(n)), GRB.MAXIMIZE)
elif mode == "max_sharpe":
    m.setObjective(t, GRB.MAXIMIZE)
elif mode == "min_var":
    m.setObjective(var_norm, GRB.MINIMIZE)
else:
    print("CHOOSE VALID MODE")
    raise ValueError

m.optimize()


Set parameter MIPGap to value 0.01
Set parameter TimeLimit to value 60
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Fedora Linux 42 (Workstation Edition)")

CPU model: AMD Ryzen 9 5900X 12-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Non-default parameters:
TimeLimit  60
MIPGap  0.01

Optimize a model with 64 rows, 255 columns and 135 nonzeros
Model fingerprint: 0xb1b5f5ad
Model has 20 quadratic constraints
Model has 108 simple general constraints
  108 PWL
Model has 46 general nonlinear constraints (459 nonlinear terms)
Variable types: 255 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  QMatrix range    [3e-01, 5e+02]
  QLMatrix range   [1e-03, 3e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e-01, 5e-01]
  RHS range        [4e-02, 3e+00]
  QRHS range       [1e-02, 5e+01]
  PWLCon x range   [0e+00, 5e+00]
  PWLCon y range   [5e-07

### Extracting + cleaning resultant weights and deltas

In [222]:
var = m.getVars()

weights = [float(v.X) for v in var if "weight" in v.VarName]
deltas = [float(v.X) for v in var if "Delta" in v.VarName]

# REMOVING WEIGHTS BELOW 0.1%
w = np.array([w if w > 0.001 else 0 for w in weights ])
w /= w.sum()

print("Ticker, Weight, Delta")
for i, ticker in enumerate(all_stocks):
    if w[i]:
        print(f"{ticker}: \t {round(w[i]*100, 2)}% \t {deltas[i]}")

Ticker, Weight, Delta
GOOG: 	 22.22% 	 0.5
WMT: 	 55.69% 	 0.1
XOM: 	 22.1% 	 0.4946888095175559


### Generating graph for portfolio with and without options

In [223]:
dfs = []
num_quarters = 21

for i, ticker in enumerate(all_stocks):
    if not weights[i]: continue
    

    df = pd.DataFrame(columns =['Open', 'price_with_put'])
    for q in range(num_quarters):
        period_start = (end + relativedelta(months=3*q))
        period_end = (end + relativedelta(months=3*(q+1)))

        hist = yf.Ticker(ticker).history(start=period_start, end=period_end, interval="1d")[['Open']]

        
        hist["price_with_put"] = hist.Open - hist.Open.iloc[0] + previous_value if q else hist.Open

        
        strike = get_strike_from_delta(deltas[i], hist.Open.iloc[0]*2, period_start.strftime("%Y-%m-%d"), period_end.strftime("%Y-%m-%d"), r, hist.Open.iloc[0], np.sqrt(Sigma[i,i])*2)
        
        num_days = len(hist.index)

        put_value = np.array([black.black_put(hist.Open.iloc[d], strike, (period_end-hist.reset_index().Date.apply(lambda x: x.replace(tzinfo=None)).iloc[d]).days/365.25, r, np.sqrt(Sigma[i,i])*2) for d in range(num_days)])
        
        put_value -= put_value[0]
        hist.price_with_put += put_value
        
        previous_value = hist.price_with_put.iloc[-1]
        df = hist if not len(df.index) else pd.concat([df, hist])

    df.Open *= w[i]/df.iloc[0].Open
    df.price_with_put *= w[i]/df.iloc[0].price_with_put
    dfs.append(df)

init_df = dfs[0]

for i, d in enumerate(dfs):
    if i:
        init_df = init_df.add(d, fill_value=0)

### Adding comparison + normalizing returns

In [224]:
benchmark_ticker = 'SPY'

init_df[benchmark_ticker] = yf.Ticker(benchmark_ticker) \
                            .history(start = end, 
                                     end = end+relativedelta(months=3*(num_quarters+1))).Open

init_df /= init_df.iloc[0]/100

px.line(init_df)

In [232]:
def get_metric(price_df, metric, start, num_quarters, r):
    returns = np.zeros(num_quarters)
    for q in range(num_quarters):
        period_start = (start + relativedelta(months=3*q))
        period_end = (start + relativedelta(months=3*(q+1)))

        slice_df = price_df.loc[(price_df.index >= period_start) & (price_df.index < period_end)]

        returns[q] = (slice_df.iloc[-1]-slice_df.iloc[0])/slice_df.iloc[0]

    if metric == "max_mu":   
        return returns.mean()
    elif metric == "max_sharpe":
        return (returns.mean()-r)/np.std(returns)
    elif metric == "min_var":
        return np.std(returns)
    else:
        print("CHOOSE VALID MODE")
        raise ValueError
    

def remove_tz(df):
    df = df.reset_index()
    df.Date = df.Date.apply(lambda x: x.replace(tzinfo=None))
    return df.set_index("Date")


print(f"Mode: {mode}")
print("Column, Mu, Var, Sharpe")

for c in init_df.columns:
    metrics = []
    for metric in ['max_mu', 'min_var', 'max_sharpe']:
        metrics.append(str(get_metric(remove_tz(init_df)[c], metric, end, num_quarters, r)))

    print(f"{c} \t {'\t'.join(metrics)}")



Mode: min_var
Column, Mu, Var, Sharpe
Open 	 0.0594278836855152	0.06487473436544111	0.7454811747503591
price_with_put 	 0.054040751164241056	0.055994693014963154	0.7674970314348936
SPY 	 0.031459202569856994	0.07100633939867997	0.28721677871887963


In [217]:
metric = get_metric(remove_tz(init_df)['price_with_put'], mode, end, num_quarters, r)

metric.mean()

np.float64(0.1669496401341403)

In [ ]:
array = []